# Step-by-Step 学习版：Gymnasium Walker2d-v4 中的 *Implementation Matters*

这个 notebook 是一个**教学版、可运行、单文件自包含**的学习笔记，目标是帮助你按顺序理解并跑通论文
*Implementation Matters in Deep Policy Gradients: A Case Study on PPO and TRPO* 的核心实现细节，同时把运行时固定在 `gymnasium.make("Walker2d-v4")`。

设计原则：

- 按学习顺序拆分，而不是把全部代码塞成几大块。
- 所有核心类和函数都在 notebook 内部实现，不依赖原仓库的任何自定义模块。
- 对关键技术点补充 Markdown 解释，包括数学习题级别的公式说明。
- 保留论文/官方仓库的主要算法结构与超参数组织方式，但不再声称这是对 `Walker2d-v2` 的精确环境复现。


## 学习路线图

你可以按照下面的顺序阅读和执行：

1. 环境与运行边界：先确认 notebook 现在固定使用 `gymnasium` 的 `Walker2d-v4`。
2. 论文核心思想：理解论文为什么强调 implementation matters。
3. 实验配置：读懂这里保留的 Walker 预设及其超参数。
4. 归一化与工具函数：状态标准化、奖励标准化、折扣累计、共轭梯度等底层组件。
5. 环境封装：把 Gymnasium 环境包装成可直接接入训练器的接口。
6. 模型结构：Gaussian actor 与 value critic。
7. 轨迹、GAE、value loss：从采样数据构造训练目标。
8. PPO 与 TRPO 更新：看清 surrogate、clipping、KL 约束和 line search。
9. 训练器：把采样、更新、日志组织成一个完整训练循环。
10. 实验运行：单次训练、多 seed 训练、曲线绘制和结果检查。

推荐使用方式：

- 第一次阅读时按顺序一个单元一个单元执行。
- 第二次运行时，把重点放在 `config`、`collect_rollouts`、`compute_advantages_and_returns`、`ppo_step` / `trpo_step` 这几个环节。
- 如果你在写课程报告，可以直接复用这里的 Markdown 结构作为讲解骨架。


## 1. 环境与运行边界

论文原始实验使用的是 **`Walker2d-v2`**，但这个 notebook 现在**明确固定为 `gymnasium` 的 `Walker2d-v4`**。
为了避免“环境版本悄悄变化，结果却看起来还能跑”的问题，这里坚持以下原则：

- 只导入 `gymnasium`
- 默认只尝试创建 `gymnasium.make("Walker2d-v4")`
- 不会静默回退到旧版 `gym`
- 也不会自动切换到 `Walker2d-v5`

这很重要，因为：

- MuJoCo 环境版本差异会改变动力学实现、episode 结束条件或 reward 细节
- 论文本身讨论的就是“看似小的实现差异会导致大结果差异”

运行前建议你先准备好兼容的运行时。典型上需要：

- `torch`
- `gymnasium`
- 可创建 `Walker2d-v4` 的 MuJoCo 运行时，通常可通过安装 `gymnasium[mujoco]` 获得

如果你后续要把结果和论文表格直接比较，请明确说明这里的环境目标是 `Walker2d-v4`，不是论文原始的 `Walker2d-v2`。


In [ ]:
from __future__ import annotations

import math
import random
import time
from dataclasses import asdict, dataclass, replace
from pprint import pprint
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.nn.utils import parameters_to_vector as flatten_parameters
    from torch.nn.utils import vector_to_parameters as assign_parameters
except Exception as exc:
    raise ImportError(
        "需要先安装 PyTorch 才能运行本 notebook。"
    ) from exc

try:
    import gymnasium
except Exception as exc:
    raise ImportError(
        "需要安装 gymnasium，并确保当前环境具备 Walker2d-v4 所需的 MuJoCo 依赖。"
    ) from exc

torch.set_default_dtype(torch.float32)
np.set_printoptions(precision=4, suppress=True)

print(f"Gymnasium version: {gymnasium.__version__}")
print(f"PyTorch version: {torch.__version__}")


## 2. 论文在做什么？

这篇论文的中心观点不是“发明一个全新的 policy gradient 算法”，而是：

> 很多深度强化学习论文里，看起来只是工程实现上的小差异，实际上足以显著改变最终结果。

对 `PPO` 和 `TRPO` 来说，影响结果的细节包括：

- observation normalization
- reward / return normalization
- value clipping
- orthogonal initialization
- learning rate annealing
- gradient clipping
- entropy bonus 的符号和系数
- KL 约束设置

论文不是只给出一个“最优算法”，而是系统比较这些实现细节如何组合后改变性能。因此，本 notebook 也不只保留一种配置，而是内置多个 Walker 预设，并把它们统一跑在 `Walker2d-v4` 上：

- `walker_ppo`
- `walker_trpo`
- `walker_ppom`
- `walker_trpo_plus`
- `walker_ppo_noclip`

其中默认学习入口是 `walker_ppo`。


## 3. Walker2d-v4 实验配置

下面先把实验配置抽象成一个 dataclass。这样做的好处是：

- 你可以非常清楚地看到每个超参数属于哪一部分
- 后面切换预设时，不需要来回读零散变量
- 课程项目中写“实验设置”章节时，可以直接引用这里的组织方式

这里的字段基本沿用官方仓库的 Walker 配置文件以及共享基础配置，但运行时目标改成了 `gymnasium` 的 `Walker2d-v4`。

额外说明：

- `seed` 是教学版 notebook 为了可审计性额外显式加入的字段
- 它不改变 PPO/TRPO 公式，只是让实验更容易复现和记录


In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    """集中定义单次 Walker2d-v4 实验的全部超参数，便于复现实验配置与后续审计。"""

    env_id: str = "Walker2d-v4"
    mode: str = "ppo"
    num_actors: int = 1
    rollout_steps: int = 2048
    train_steps: int = 488
    num_minibatches: int = 32
    ppo_epochs: int = 10
    val_epochs: int = 10
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_eps: float = 0.2
    clip_val_eps: float = 0.2
    entropy_coeff: float = 0.0
    ppo_lr: float = -1.0
    ppo_lr_adam: float = 3e-4
    val_lr: float = 1.5e-4
    max_kl: float = 0.01
    max_kl_final: float = 0.01
    fisher_frac_samples: float = 0.1
    cg_steps: int = 10
    damping: float = 0.1
    max_backtrack: int = 10
    norm_states: bool = True
    norm_rewards: str = "returns"
    clip_rewards: Optional[float] = 10.0
    clip_observations: Optional[float] = 10.0
    initialization: str = "orthogonal"
    value_clipping: bool = True
    anneal_lr: bool = True
    clip_grad_norm: float = -1.0
    adam_eps: float = 1e-5
    hidden_sizes: Tuple[int, int] = (64, 64)
    seed: Optional[int] = 0
    device: str = "cpu"
    advanced_logging: bool = True
    log_every: int = 10


BASE_WALKER_REPRO = ExperimentConfig()

CONFIG_PRESETS: Dict[str, ExperimentConfig] = {
    "walker_ppo": replace(
        BASE_WALKER_REPRO,
        env_id="Walker2d-v4",
        mode="ppo",
        norm_rewards="returns",
        initialization="orthogonal",
        anneal_lr=True,
        value_clipping=True,
        ppo_lr_adam=4e-4,
        val_lr=3e-4,
        advanced_logging=True,
    ),
    "walker_trpo": replace(
        BASE_WALKER_REPRO,
        env_id="Walker2d-v4",
        mode="trpo",
        norm_rewards="none",
        initialization="xavier",
        anneal_lr=False,
        value_clipping=False,
        max_kl=0.04,
        max_kl_final=0.04,
        val_lr=3e-4,
        clip_rewards=None,
        clip_observations=None,
        advanced_logging=False,
    ),
    "walker_ppom": replace(
        BASE_WALKER_REPRO,
        env_id="Walker2d-v4",
        mode="ppo",
        norm_rewards="none",
        initialization="xavier",
        anneal_lr=False,
        value_clipping=False,
        ppo_lr_adam=1e-4,
        val_lr=2e-4,
        clip_rewards=None,
        clip_observations=None,
        advanced_logging=True,
    ),
    "walker_trpo_plus": replace(
        BASE_WALKER_REPRO,
        env_id="Walker2d-v4",
        mode="trpo",
        norm_rewards="returns",
        initialization="orthogonal",
        anneal_lr=False,
        value_clipping=False,
        max_kl=0.07,
        max_kl_final=0.04,
        val_lr=1e-4,
        clip_grad_norm=1.0,
        advanced_logging=True,
    ),
    # 这里沿用 Walker2d-v2 论文 Table 4 给出的 PPO-NoClip 观测截断 [-30, 30]。
    "walker_ppo_noclip": replace(
        BASE_WALKER_REPRO,
        env_id="Walker2d-v4",
        mode="ppo",
        norm_rewards="rewards",
        initialization="orthogonal",
        anneal_lr=True,
        value_clipping=False,
        clip_eps=1e32,
        ppo_lr_adam=0.725e-4,
        entropy_coeff=-0.01,
        clip_rewards=30.0,
        clip_observations=30.0,
        gae_lambda=0.85,
        val_lr=6e-4,
        clip_grad_norm=0.1,
        advanced_logging=True,
    ),
}


def make_config(preset_name: str = "walker_ppo", **overrides) -> ExperimentConfig:
    """从预设配置生成实验参数，并允许用显式关键字覆盖默认超参数。"""

    if preset_name not in CONFIG_PRESETS:
        raise KeyError(f"Unknown preset: {preset_name}. Available presets: {sorted(CONFIG_PRESETS)}")
    return replace(CONFIG_PRESETS[preset_name], **overrides)


def print_config(config: ExperimentConfig) -> None:
    """以易读格式打印当前实验配置，便于在启动训练前人工核对参数。"""

    pprint(asdict(config), sort_dicts=False)


print("Available presets:", ", ".join(CONFIG_PRESETS.keys()))
print_config(make_config("walker_ppo"))


## 4. 归一化与底层工具函数

这一部分是整篇论文非常重要的“implementation matters”来源。

### 4.1 Observation normalization

状态标准化形式为

$$
\tilde s_t = \mathrm{clip}\left(
    \frac{s_t - \mu_t}{\sigma_t + 10^{-8}},
    -c_{\mathrm{obs}}, c_{\mathrm{obs}}
\right).
$$

其中 $\mu_t, \sigma_t$ 不是固定数据集统计量，而是在线更新的运行均值和标准差。

### 4.2 Return normalization

当 `norm_rewards="returns"` 时，论文/官方实现采用的是 OpenAI 风格的做法：

$$
G_t = \gamma G_{t-1} + r_t,
\qquad
\tilde r_t = \frac{r_t}{\mathrm{Std}(G_{1:t}) + 10^{-8}}.
$$

注意这里并没有把 reward 减去 return 的均值，而只是除以 return 的标准差。

### 4.3 TRPO 的底层数值工具

TRPO 的自然梯度更新需要：

- Fisher-vector product
- conjugate gradient
- backtracking line search

这些都不是“只要会调包就行”的细节，反而正是论文强调的关键实现组成。


In [ ]:
def set_global_seed(seed: Optional[int]) -> None:
    """统一设置 Python、NumPy 和 PyTorch 的随机种子，降低多次运行之间的随机差异。"""

    if seed is None:
        return
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device(device_name: str) -> torch.device:
    """根据用户指定和硬件可用性选择实际使用的计算设备。"""

    if device_name == "cuda" and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def to_tensor(x, device: torch.device) -> torch.Tensor:
    """把输入数据转换为指定设备上的 float32 Tensor，统一后续数值计算接口。"""

    return torch.as_tensor(x, dtype=torch.float32, device=device)


def flatten_first_two_dims(tensor: Optional[torch.Tensor]) -> Optional[torch.Tensor]:
    """将 `(num_actors, T, ...)` 形式的张量展平成 `(num_actors*T, ...)`，用于小批量优化。"""

    if tensor is None:
        return None
    if tensor.ndim < 2:
        raise ValueError("Expected tensor with at least two dimensions to flatten.")
    new_shape = (tensor.shape[0] * tensor.shape[1],) + tuple(tensor.shape[2:])
    return tensor.contiguous().view(*new_shape)


def minibatch_splits(size: int, num_minibatches: int) -> List[np.ndarray]:
    """随机打乱样本索引并切分成多个 minibatch，供 PPO/TRPO 的 epoch 内更新使用。"""

    indices = np.arange(size)
    np.random.shuffle(indices)
    return [split for split in np.array_split(indices, num_minibatches) if len(split) > 0]


def discount_path(path: torch.Tensor, discount: float) -> torch.Tensor:
    """沿一条轨迹从后向前计算折扣累计和，用于回报和 GAE 优势的递推。"""

    running = torch.zeros((), dtype=path.dtype, device=path.device)
    discounted = []
    for value in reversed(path):
        running = value + discount * running
        discounted.append(running)
    return torch.stack(list(reversed(discounted)))


def get_path_indices(not_dones: torch.Tensor) -> List[Tuple[int, int, int]]:
    """根据 `not_dones` 标记恢复每条完整轨迹的起止位置，避免跨 episode 误做折扣累计。"""

    indices: List[Tuple[int, int, int]] = []
    # 获取总的时间步数（例如 100 步）
    num_timesteps = not_dones.shape[1]
    # 外层循环：遍历每一个并行的环境（Actor）
    for actor in range(not_dones.shape[0]):
        last_index = 0 # 记录当前这条轨迹的起始位置
        # 内层循环：顺着时间线寻找结束标志（done），每当找到一个 done 就把当前轨迹的起止位置记录下来，并更新起始位置为下一个时间步
        for t in range(num_timesteps):
            # 如果 not_dones 是 0.0，说明在第 t 步游戏结束了（Game Over）
            if float(not_dones[actor, t].item()) == 0.0:
                # 记录这条轨迹：(当前Actor编号, 起点, 终点)
                indices.append((actor, last_index, t + 1))
                # 更新下一条轨迹的起点为当前结束点的下一步
                last_index = t + 1
        # 边界处理：如果在跑完 num_timesteps 后，游戏还没结束（被硬性截断了）        
        if last_index != num_timesteps:
            # 最后这段不完整的轨迹也记录下来
            indices.append((actor, last_index, num_timesteps))
    return indices


def select_prob_dists(pds, selected: Optional[np.ndarray] = None, detach: bool = True):
    """从策略分布参数中选取指定样本，并按需对结果执行 `detach`。"""

    # 连续动作空间（如方向盘转角、油门大小）：网络通常输出高斯分布的均值（Mean）和标准差（Std）。此时 pds 是一个 Tuple，例如 (mean_tensor, std_tensor)。
    # selected：一个包含索引的 Numpy 数组，相当于“掩码”或“下标”。用于从收集到的海量经验中，抽取出当前要训练的那一小批数据（Mini-batch）。
    # detach：一个布尔值，指示是否要对选取的张量执行 `detach()` 操作，以切断计算图，防止梯度回传到原始网络参数。

    # 为什么需要 selected？ 因为我们在更新时，是把几千步的轨迹打乱，每次抽取一个 mini-batch（比如 64 步）来算梯度，所以要用 selected 抽出这 64 步对应的 old_pds。
    # 例如，如果 pds 是一个 Tuple (mean_tensor, std_tensor)，我们就需要对 mean_tensor 和 std_tensor 都进行索引选取，得到对应 mini-batch 的 mean 和 std。然后根据 detach 参数决定是否要对这两个张量执行 detach()。
    # 为什么需要 detach=True？ 因为“旧策略”是一个固定不变的参考基准，我们只需要它的数值来计算比率（Ratio），绝对不能让梯度流回旧策略的网络中去。使用 detach 可以保证旧参数的绝对安全，同时节省显存。

    if isinstance(pds, tuple):
        first = pds[0][selected] if selected is not None else pds[0]
        second = pds[1]
        if detach:
            return first.detach(), second.detach()
        return first, second
    output = pds[selected] if selected is not None else pds
    return output.detach() if detach else output


class RunningStat:
    """维护流式均值和方差统计量，为状态标准化和奖励标准化提供运行时统计。"""
    # 该类实现了 Welford 算法，可以在线更新均值和方差估计，避免数值不稳定和内存占用过大问题。
    # 更新均值： 当前均值等于旧均值加上“新样本与旧均值的差值”平摊到 $n$ 个样本上。
    # 更新方差： 当前二阶矩等于旧二阶矩加上“新样本与旧均值的差值”乘以“新样本与新均值的差值”，这样就能在线更新方差估计，而不需要存储所有样本。
    # 这里 $S$ 代表误差平方和。Welford 给出了一个极其优雅且数值稳定的更新公式：$$S_n = S_{n-1} + (x_n - M_{n-1})(x_n - M_n)$$
    # $$\sigma^2_n = \frac{S_n}{n - 1}$$
    def __init__(self, shape):
        """初始化流式统计器内部的样本数、均值和二阶矩缓存。"""

        self._n = 0
        self._M = np.zeros(shape, dtype=np.float64)
        self._S = np.zeros(shape, dtype=np.float64)

    def push(self, x):
        """把一个新样本并入当前统计量，在线更新均值与方差估计。"""

        x = np.asarray(x, dtype=np.float64)
        if x.shape != self._M.shape:
            raise ValueError(f"Expected shape {self._M.shape}, got {x.shape}")
        self._n += 1
        if self._n == 1:
            self._M[...] = x
        else:
            old_mean = self._M.copy()
            self._M[...] = old_mean + (x - old_mean) / self._n
            self._S[...] = self._S + (x - old_mean) * (x - self._M)

    @property
    def n(self):
        """返回已经累计到统计器中的样本数量。"""

        return self._n

    @property
    def mean(self):
        """返回当前运行均值估计。"""

        return self._M

    @property
    def var(self):
        """返回当前运行方差估计。"""

        return self._S / (self._n - 1) if self._n > 1 else np.square(self._M)

    @property
    def std(self):
        """返回当前运行标准差估计。"""

        return np.sqrt(self.var)


class Identity:
    """实现一个不做任何变换的过滤器，便于统一包装状态和奖励处理流水线。"""
    # # 在初始化流水线时，统一分配好“工具”
    # if use_normalization:
    #     self.state_filter = Normalizer(shape) # 比如基于 RunningStat 实现的类
    # else:
    #     self.state_filter = Identity()        # 占位符：什么都不做


    def __call__(self, x, *args, **kwargs):
        """直接返回输入本身，用作归一化链条中的空操作占位符。"""

        return x

    def reset(self):
        """重置空过滤器；该实现中无需真正执行任何状态更新。"""

        pass


class RewardFilter:
    """按论文实现中的 OpenAI 风格返回值标准差对奖励做缩放。"""
    # 追踪折扣累计回报 $R$ 的标准差 $\sigma_R$，然后用它来除以单步奖励 $r_t$。这样可以确保最终计算出的长期回报方差约为 $1$，使得优化过程更稳定。
    # 这里没有减去均值。在强化学习中，如果我们减去奖励的均值，可能会把原本微小的正奖励（鼓励生存）变成负奖励（鼓励自杀），从而彻底改变游戏的目标（Reward Shaping 陷阱）。因此，奖励标准化通常只做缩放，不做平移。

    def __init__(self, prev_filter, shape, gamma: float, clip: Optional[float] = None):
        """初始化奖励过滤器、返回累计器和对应的运行统计量。"""

        self.prev_filter = prev_filter
        self.gamma = gamma
        self.rs = RunningStat(shape)
        self.ret = np.zeros(shape, dtype=np.float64)
        self.clip = clip

    def __call__(self, x, **kwargs):
        """更新折扣 return 的统计并用其标准差缩放当前奖励。"""

        x = self.prev_filter(x, **kwargs)
        self.ret = self.ret * self.gamma + x
        self.rs.push(self.ret)
        x = x / (self.rs.std + 1e-8)
        if self.clip is not None:
            x = np.clip(x, -self.clip, self.clip)
        return x

    def reset(self):
        """在 episode 重置时清空 return 累计，并重置前序过滤器。"""

        self.ret = np.zeros_like(self.ret)
        self.prev_filter.reset()


class ZFilter:
    """使用运行均值和标准差对输入执行在线 z-score 标准化，可选截断。"""
    # 处理状态 (State)： 用 ZFilter。既要减均值 (Center)，又要除以标准差 (Scale)。因为状态的绝对数值大小通常没意义，相对大小才有意义。

    def __init__(
        self,
        prev_filter,
        shape,
        center: bool = True,
        scale: bool = True,
        clip: Optional[float] = None,
    ):
        """初始化 z-score 过滤器的中心化、缩放和截断等行为参数。"""

        self.prev_filter = prev_filter
        self.center = center
        self.scale = scale
        self.clip = clip
        self.rs = RunningStat(shape)

    def __call__(self, x, **kwargs):
        """把新输入并入统计量后执行中心化、缩放和可选截断。"""

        x = self.prev_filter(x, **kwargs)
        self.rs.push(x)
        if self.center:
            x = x - self.rs.mean
        if self.scale:
            if self.center:
                x = x / (self.rs.std + 1e-8)
            else:
                diff = x - self.rs.mean
                x = diff / (self.rs.std + 1e-8) + self.rs.mean
        if self.clip is not None:
            x = np.clip(x, -self.clip, self.clip)
        return x

    def reset(self):
        """在环境重置时同步重置前序过滤器状态。"""

        self.prev_filter.reset()


def orthogonal_init_(tensor: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """对权重张量执行正交初始化，以匹配官方实现中的网络初始化方案。"""

    if tensor.ndim < 2:
        raise ValueError("Orthogonal initialization requires a tensor with ndim >= 2.")
    nn.init.orthogonal_(tensor, gain=gain)
    return tensor


def initialize_module(module: nn.Module, initialization: str, scale: float = math.sqrt(2.0)) -> None:
    """按照论文/官方实现指定的初始化方式批量初始化模块参数。"""

    for param in module.parameters():
        if initialization == "normal":
            param.data.normal_(0.01)
        elif initialization == "xavier":
            if param.ndim >= 2:
                nn.init.xavier_uniform_(param.data)
            else:
                param.data.zero_()
        elif initialization == "orthogonal":
            if param.ndim >= 2:
                orthogonal_init_(param.data, gain=scale)
            else:
                param.data.zero_()
        else:
            raise ValueError(f"Unknown initialization scheme: {initialization}")


def conjugate_gradient(fvp_fn, b: torch.Tensor, nsteps: int) -> torch.Tensor:
    """使用共轭梯度法近似求解 Fisher 线性系统，是 TRPO 自然梯度步的核心子程序。"""

    x = torch.zeros_like(b)
    r = b.clone()
    p = b.clone()
    new_rnorm = torch.dot(r, r)
    for _ in range(nsteps):
        rnorm = new_rnorm
        fvp = fvp_fn(p)
        alpha = rnorm / torch.dot(p, fvp)
        x = x + alpha * p
        r = r - alpha * fvp
        new_rnorm = torch.dot(r, r)
        beta = new_rnorm / rnorm
        p = r + beta * p
    return x


def backtracking_line_search(
    improvement_fn,
    full_step: torch.Tensor,
    expected_improve_rate,
    num_tries: int = 10,
    accept_ratio: float = 0.1,
) -> torch.Tensor:
    """通过逐步缩小候选步长寻找满足改进率和约束条件的最终更新步。"""

    expected_improve_rate = float(expected_improve_rate)
    for i in range(num_tries):
        scaling = 2 ** (-i)
        candidate = full_step * scaling
        improve = improvement_fn(candidate)
        expected_improve = expected_improve_rate * scaling
        if improve / expected_improve > accept_ratio and improve > 0:
            return candidate
    return torch.zeros_like(full_step)


In [ ]:
# 这个小单元不是训练必须步骤，只是为了帮助理解运行统计是如何变化的。
demo_rs = RunningStat(shape=())
for value in [1.0, 2.0, 3.0, 4.0]:
    demo_rs.push(value)
print("RunningStat mean:", demo_rs.mean)
print("RunningStat std :", demo_rs.std)

demo_z = ZFilter(Identity(), shape=(), clip=5.0)
print("ZFilter demo outputs:", [float(demo_z(v)) for v in [1.0, 2.0, 3.0, 4.0]])


In [ ]:
demo_z(0.2)

## 5. 环境封装

训练器不应该直接和原始 Gym API 的各种版本差异纠缠在一起。  
因此我们定义一个 `WalkerEnv`，负责：

- 统一 `reset` / `step` 的返回格式
- 在环境层应用状态标准化和奖励标准化
- 记录真实 episode reward 和 episode length

这样做的好处是，后面的训练器逻辑可以只关注“采样到了什么张量”，而不用关心底层 API 风格。


In [ ]:
class WalkerEnv:
    """封装 Gymnasium Walker2d-v4 环境，并叠加状态标准化与奖励标准化逻辑。"""

    def __init__(self, config: ExperimentConfig):
        """创建原始环境并根据实验配置装配状态/奖励过滤器。"""

        self.config = config
        try:
            self.env = gymnasium.make(config.env_id)
        except Exception as exc:
            raise RuntimeError(
                "需要可成功创建的 gymnasium.make('Walker2d-v4') 运行时。若缺少 MuJoCo，请安装 `gymnasium[mujoco]`。"
            ) from exc

        if not isinstance(self.env.action_space, gymnasium.spaces.Box):
            raise TypeError("Walker2d-v4 is expected to have a continuous Box action space.")
        if len(self.env.observation_space.shape) != 1:
            raise TypeError("This notebook expects a flat state vector.")

        self.num_features = int(self.env.observation_space.shape[0])
        self.num_actions = int(self.env.action_space.shape[0])

        self.state_filter = Identity()
        if config.norm_states:
            self.state_filter = ZFilter(
                self.state_filter,
                shape=(self.num_features,),
                clip=config.clip_observations,
            )

        self.reward_filter = Identity()
        if config.norm_rewards == "rewards":
            self.reward_filter = ZFilter(
                self.reward_filter,
                shape=(),
                center=False,
                clip=config.clip_rewards,
            )
        elif config.norm_rewards == "returns":
            self.reward_filter = RewardFilter(
                self.reward_filter,
                shape=(),
                gamma=config.gamma,
                clip=config.clip_rewards,
            )
        elif config.norm_rewards != "none":
            raise ValueError(f"Unknown reward normalization mode: {config.norm_rewards}")

        self.total_true_reward = 0.0
        self.counter = 0.0

    def reset(self, seed: Optional[int] = None):
        """重置环境与过滤器状态，并返回经过相同预处理后的初始观测。"""

        if seed is None:
            obs, _ = self.env.reset()
        else:
            obs, _ = self.env.reset(seed=seed)

        self.total_true_reward = 0.0
        self.counter = 0.0
        self.state_filter.reset()
        self.reward_filter.reset()
        return self.state_filter(np.asarray(obs, dtype=np.float32), reset=True)

    def step(self, action: np.ndarray):
        """执行一步环境交互，记录真实 episode 回报并返回过滤后的观测与奖励。"""

        obs, reward, terminated, truncated, info = self.env.step(action)
        done = bool(terminated or truncated)
        obs = np.asarray(obs, dtype=np.float32)

        self.total_true_reward += float(reward)
        self.counter += 1.0
        filtered_reward = self.reward_filter(float(reward))

        info = dict(info)
        if done:
            info["done"] = (self.counter, self.total_true_reward)
        return self.state_filter(obs), float(filtered_reward), done, info


In [ ]:
# 这是一个环境检查入口。默认关闭，等你确认本机 MuJoCo 运行时已经准备好后再打开。
RUN_ENV_SANITY_CHECK = False

if RUN_ENV_SANITY_CHECK:
    demo_config = make_config("walker_ppo")
    demo_env = WalkerEnv(demo_config)
    obs = demo_env.reset(seed=demo_config.seed)
    print("obs shape    :", np.asarray(obs).shape)
    print("action dim   :", demo_env.num_actions)
    print("feature dim  :", demo_env.num_features)
    print("env created  :", demo_config.env_id)
else:
    print("环境检查默认关闭。需要验证 Gymnasium / MuJoCo 运行时时，将 RUN_ENV_SANITY_CHECK 设为 True。")


## 6. 模型结构：Gaussian Actor 与 Value Critic

Walker2d-v4 是连续控制任务，因此策略网络输出的是**对角高斯分布**参数，而不是离散动作概率。

### 6.1 Policy

给定状态 $s$，actor 输出

$$
\mu_\theta(s), \quad \sigma_\theta
$$

其中该实现采用的是：

- 用 MLP 输出 $\mu_\theta(s)$
- 用一个**与状态无关**的可学习参数 `log_stdev` 表示全局标准差

采样动作：

$$
a \sim \mathcal{N}(\mu_\theta(s), \mathrm{diag}(\sigma_\theta^2))
$$

### 6.2 Value Function

critic 使用两层 `tanh` MLP，输出标量 $V_\phi(s)$。

### 6.3 初始化

论文特别强调初始化方式的影响。这里保留：

- hidden layers: 正交初始化，gain = $\sqrt{2}$
- value head: gain = 1.0
- policy mean head: gain = 0.01


In [ ]:
class ValueDenseNet(nn.Module):
    """实现论文中使用的两层 tanh MLP critic，用于估计状态价值函数。"""

    def __init__(self, state_dim: int, initialization: str, hidden_sizes: Sequence[int] = (64, 64)):
        """构建 value network 的隐藏层和输出层，并完成参数初始化。"""

        super().__init__()
        self.activation = nn.Tanh()
        self.affine_layers = nn.ModuleList()

        prev_dim = state_dim
        for hidden_dim in hidden_sizes:
            layer = nn.Linear(prev_dim, hidden_dim)
            initialize_module(layer, initialization)
            self.affine_layers.append(layer)
            prev_dim = hidden_dim

        self.final = nn.Linear(prev_dim, 1)
        initialize_module(self.final, initialization, scale=1.0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """将状态输入映射为标量价值估计。"""

        for layer in self.affine_layers:
            x = self.activation(layer(x))
        return self.final(x)


class CtsPolicy(nn.Module):
    """实现 Walker2d-v4 使用的对角高斯连续策略网络。"""

    def __init__(self, state_dim: int, action_dim: int, initialization: str, hidden_sizes: Sequence[int] = (64, 64)):
        """构建策略网络的均值头和独立的 log standard deviation 参数。"""

        super().__init__()
        self.action_dim = action_dim
        self.activation = nn.Tanh()
        self.affine_layers = nn.ModuleList()

        prev_dim = state_dim
        for hidden_dim in hidden_sizes:
            layer = nn.Linear(prev_dim, hidden_dim)
            initialize_module(layer, initialization)
            self.affine_layers.append(layer)
            prev_dim = hidden_dim

        self.final_mean = nn.Linear(prev_dim, action_dim)
        initialize_module(self.final_mean, initialization, scale=0.01)

        # 官方实现中，log_stdev 不是网络输出，而是一个独立的可学习参数。
        self.log_stdev = nn.Parameter(torch.zeros(action_dim))

    def forward(self, x: torch.Tensor):
        """根据状态输出高斯策略的均值向量与标准差向量。"""

        for layer in self.affine_layers:
            x = self.activation(layer(x))
        means = self.final_mean(x)
        std = torch.exp(self.log_stdev)
        return means, std

    def sample(self, p):
        """从当前高斯策略中采样动作，用于环境交互。"""

        means, std = p
        return (means + torch.randn_like(means) * std).detach()

    def get_loglikelihood(self, p, actions: torch.Tensor) -> torch.Tensor:
        """计算给定动作在当前高斯策略下的对数似然。"""

        mean, std = p
        nll = (
            0.5 * ((actions - mean) / std).pow(2).sum(dim=-1)
            + 0.5 * math.log(2.0 * math.pi) * actions.shape[-1]
            + self.log_stdev.sum(dim=-1)
        )
        return -nll

    def calc_kl(self, p, q) -> torch.Tensor:
        """计算两组对角高斯策略之间的 KL 散度，用于 TRPO 约束与日志记录。"""

        p_mean, p_std = p
        q_mean, q_std = q
        p_var = p_std.pow(2)
        q_var = q_std.pow(2)
        diff = q_mean - p_mean
        d = q_mean.shape[1]
        log_quot_frac = torch.log(q_var).sum() - torch.log(p_var).sum()
        trace_term = (p_var / q_var).sum()
        quadratic = ((diff / q_var) * diff).sum(dim=1)
        return 0.5 * (log_quot_frac - d + trace_term + quadratic)

    def entropies(self, p) -> torch.Tensor:
        """计算当前对角高斯策略的熵，用于 PPO 的熵正则项。"""

        _, std = p
        detp = torch.exp(torch.log(std).sum())
        d = std.shape[0]
        entropy = torch.log(detp) + 0.5 * d * (1.0 + math.log(2.0 * math.pi))
        return entropy


In [ ]:
# 这个单元帮助你确认网络的输入输出维度。
fake_obs = torch.randn(4, 17)
actor = CtsPolicy(state_dim=17, action_dim=6, initialization="orthogonal")
critic = ValueDenseNet(state_dim=17, initialization="orthogonal")

actor_mean, actor_std = actor(fake_obs)
critic_value = critic(fake_obs)

print("actor mean shape:", tuple(actor_mean.shape))
print("actor std shape :", tuple(actor_std.shape))
print("critic shape    :", tuple(critic_value.shape))


## 7. 轨迹数据、GAE、PPO surrogate 与 value loss

### 7.1 GAE

定义 temporal-difference 误差：

$$
\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)
$$

则 generalized advantage estimation 为

$$
\hat A_t = \sum_{l=0}^{T-t-1} (\gamma \lambda)^l \delta_{t+l}.
$$

### 7.2 Discounted return

$$
\hat R_t = \sum_{l=0}^{T-t-1} \gamma^l r_{t+l}.
$$

### 7.3 PPO surrogate

$$
r_t(\theta) =
\exp\left(
    \log \pi_\theta(a_t \mid s_t) -
    \log \pi_{\theta_{\mathrm{old}}}(a_t \mid s_t)
\right)
$$

PPO 的核心目标为

$$
L^{\mathrm{PPO}}(\theta)
=
- \mathbb{E}_t \left[
    \min\left(
        r_t(\theta)\hat A_t,\;
        \mathrm{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat A_t
    \right)
\right]
-
c_{\mathrm{ent}}\mathbb{E}_t[\mathcal{H}(\pi_\theta)].
$$

### 7.4 Value clipping

论文和官方实现使用了 OpenAI 风格的 clipped value loss。  
令

$$
V_t^{\mathrm{target}} = V_{\mathrm{old}}(s_t) + \hat A_t
$$

并定义

$$
V_t^{\mathrm{clip}} =
V_{\mathrm{old}}(s_t) +
\mathrm{clip}\left(
    V_\phi(s_t)-V_{\mathrm{old}}(s_t),
    -\epsilon_v, \epsilon_v
\right).
$$

最终 loss 取 clipped / unclipped 中更坏的那个平方误差。


In [ ]:
@dataclass
class TrajectoryBatch:
    """保存一批 on-policy 轨迹数据及其派生量，作为训练阶段的统一数据容器。"""

    states: torch.Tensor
    rewards: torch.Tensor
    not_dones: torch.Tensor
    actions: torch.Tensor
    action_log_probs: torch.Tensor
    returns: Optional[torch.Tensor] = None
    advantages: Optional[torch.Tensor] = None
    values: Optional[torch.Tensor] = None
    unrolled: bool = False

    def unroll(self) -> "TrajectoryBatch":
        """把按 actor 和时间组织的轨迹张量展平成优化器可直接使用的样本批。"""

        if self.unrolled:
            raise ValueError("This batch is already flattened.")
        return TrajectoryBatch(
            states=flatten_first_two_dims(self.states),
            rewards=flatten_first_two_dims(self.rewards),
            not_dones=flatten_first_two_dims(self.not_dones),
            actions=flatten_first_two_dims(self.actions),
            action_log_probs=flatten_first_two_dims(self.action_log_probs),
            returns=flatten_first_two_dims(self.returns),
            advantages=flatten_first_two_dims(self.advantages),
            values=flatten_first_two_dims(self.values),
            unrolled=True,
        )


def normalize_advantages(advantages: torch.Tensor) -> torch.Tensor:
    """对优势做标准化，匹配官方 PPO/TRPO surrogate 中的优势预处理。"""

    std = advantages.std()
    if float(std.item()) == 0.0 or bool(torch.isnan(std).item()):
        raise ValueError("Advantage normalization requires a nonzero finite standard deviation.")
    return (advantages - advantages.mean()) / (advantages.std() + 1e-8)


def surrogate_reward(
    advantages: torch.Tensor,
    *,
    new_log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
    clip_eps: Optional[float] = None,
) -> torch.Tensor:
    """构造 PPO/TRPO 共用的 surrogate 项，可按需启用 PPO ratio clipping。"""

    normalized_advantages = normalize_advantages(advantages)
    ratios = torch.exp(new_log_probs - old_log_probs)
    if clip_eps is not None:
        ratios = torch.clamp(ratios, 1.0 - clip_eps, 1.0 + clip_eps)
    return ratios * normalized_advantages


def compute_advantages_and_returns(
    rewards: torch.Tensor,
    values: torch.Tensor,
    not_dones: torch.Tensor,
    gamma: float,
    gae_lambda: float,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """按官方实现细节计算 GAE 优势与折扣回报。"""

    # 官方实现里的 bootstrap 逻辑：先把 values 左移一位，再把最后一个值复制到末尾，
    # 然后整体乘以 not_dones 作为 mask。
    next_values = torch.cat([values[:, 1:], values[:, -1:]], dim=1) * not_dones
    deltas = rewards + gamma * next_values - values

    advantages = torch.zeros_like(rewards)
    returns = torch.zeros_like(rewards)
    for actor, start, end in get_path_indices(not_dones):
        advantages[actor, start:end] = discount_path(deltas[actor, start:end], gamma * gae_lambda)
        returns[actor, start:end] = discount_path(rewards[actor, start:end], gamma)
    return advantages.detach(), returns.detach()


def value_loss_gae(
    values: torch.Tensor,
    advantages: torch.Tensor,
    not_dones: torch.Tensor,
    old_values: torch.Tensor,
    clip_val_eps: float,
    value_clipping: bool,
) -> torch.Tensor:
    """计算基于 GAE target 的 value loss，并可选择 OpenAI 风格的 value clipping。"""

    value_target = (old_values + advantages).detach()
    clipped_values = old_values + torch.clamp(values - old_values, -clip_val_eps, clip_val_eps)

    valid_mask = not_dones.bool()
    unclipped_sq_error = (values - value_target)[valid_mask].pow(2)
    clipped_sq_error = (clipped_values - value_target)[valid_mask].pow(2)

    if value_clipping:
        loss_terms = torch.max(unclipped_sq_error, clipped_sq_error)
    else:
        loss_terms = unclipped_sq_error
    return loss_terms.mean()


def compute_policy_diagnostics(
    policy: CtsPolicy,
    states: torch.Tensor,
    actions: torch.Tensor,
    old_log_probs: torch.Tensor,
    old_pds,
) -> Dict[str, float]:
    """汇总策略更新后的 KL 和 importance ratio 等诊断指标。"""

    with torch.no_grad():
        new_pds = policy(states)
        new_log_probs = policy.get_loglikelihood(new_pds, actions)
        ratios = torch.exp(new_log_probs - old_log_probs)
        avg_kl = policy.calc_kl(old_pds, new_pds).mean().item()
        max_ratio = ratios.max().item()
    return {"avg_kl": avg_kl, "max_ratio": max_ratio}


In [ ]:
# 这个小例子可以帮助你检查 GAE 和 returns 的张量形状是否符合预期。
toy_rewards = torch.tensor([[1.0, 1.0, 1.0, 1.0]])
toy_values = torch.tensor([[0.2, 0.3, 0.4, 0.5]])
toy_not_dones = torch.tensor([[1.0, 1.0, 1.0, 0.0]])

toy_advantages, toy_returns = compute_advantages_and_returns(
    rewards=toy_rewards,
    values=toy_values,
    not_dones=toy_not_dones,
    gamma=0.99,
    gae_lambda=0.95,
)

print("toy advantages:", toy_advantages)
print("toy returns   :", toy_returns)


## 8. TRPO：自然梯度、KL 约束与回溯线搜索

TRPO 的目标是：

$$
\max_\theta \; \mathbb{E}_t[r_t(\theta)\hat A_t]
\quad \text{s.t.} \quad
\mathbb{E}_t\left[
    D_{\mathrm{KL}}(\pi_{\theta_{\mathrm{old}}} \| \pi_\theta)
\right] \le \delta.
$$

实际实现里不会显式构造完整 Fisher 矩阵，而是：

1. 用 KL 的 Hessian 定义 Fisher-vector product
2. 用 conjugate gradient 近似解线性系统
3. 沿自然梯度方向缩放到目标 KL 半径
4. 再用 backtracking line search 保证真实 surrogate 改进和 KL 约束都成立

这部分正是 TRPO 实现最“工程上容易写错”的地方，所以也最值得你逐句读代码。


## 9. 训练器：把采样、估值、更新连起来

训练器的职责是把前面所有积木接起来，形成一条完整训练流水线：

1. 从环境收集一段 rollout
2. 用 critic 给 rollout 中的状态估值
3. 计算 GAE 优势与 discounted returns
4. 先更新 value network
5. 再更新 policy network
6. 记录 reward、value loss、KL、ratio 等日志

你在读这个类时，建议重点看三件事：

- `collect_rollouts` 的张量组织方式
- `value_step` 与 `ppo_step` / `trpo_step` 的先后顺序
- 每个训练迭代里哪些量是 `detach` 的，哪些量会继续参与梯度传播


In [ ]:
class ImplementationMattersTrainer:
    """组织采样、GAE、value 更新、policy 更新与日志记录的训练主类。"""

    def __init__(self, config: ExperimentConfig):
        """根据实验配置创建环境、策略网络、价值网络和优化器。"""

        if config.num_actors != 1:
            raise ValueError("This notebook keeps the released single-actor rollout path (num_actors = 1).")

        self.config = config
        self.device = get_device(config.device)
        set_global_seed(config.seed)

        self.env = WalkerEnv(config)
        self.obs_dim = self.env.num_features
        self.act_dim = self.env.num_actions

        self.policy_model = CtsPolicy(
            state_dim=self.obs_dim,
            action_dim=self.act_dim,
            initialization=config.initialization,
            hidden_sizes=config.hidden_sizes,
        ).to(self.device)
        self.value_model = ValueDenseNet(
            state_dim=self.obs_dim,
            initialization=config.initialization,
            hidden_sizes=config.hidden_sizes,
        ).to(self.device)

        if not (config.ppo_lr == -1.0 or config.ppo_lr_adam == -1.0):
            raise ValueError("One of ppo_lr and ppo_lr_adam must be -1, matching the original repo.")

        if config.ppo_lr_adam != -1.0:
            optimizer_kwargs = {"lr": config.ppo_lr_adam}
            if config.adam_eps > 0:
                optimizer_kwargs["eps"] = config.adam_eps
            self.policy_optimizer = optim.Adam(self.policy_model.parameters(), **optimizer_kwargs)
        else:
            self.policy_optimizer = optim.SGD(self.policy_model.parameters(), lr=config.ppo_lr)

        self.value_optimizer = optim.Adam(self.value_model.parameters(), lr=config.val_lr, eps=1e-5)

        if config.anneal_lr:
            decay = lambda step: 1.0 - step / config.train_steps
            self.policy_scheduler = optim.lr_scheduler.LambdaLR(self.policy_optimizer, lr_lambda=decay)
            self.value_scheduler = optim.lr_scheduler.LambdaLR(self.value_optimizer, lr_lambda=decay)
        else:
            self.policy_scheduler = None
            self.value_scheduler = None

        self.current_max_kl = config.max_kl
        self.max_kl_increment = (config.max_kl_final - config.max_kl) / config.train_steps
        self.training_log: List[Dict[str, float]] = []
        self._seeded_reset_already = False

    def _reset_env(self) -> torch.Tensor:
        """统一处理首次带 seed 的环境重置与后续普通重置。"""

        seed = None
        if not self._seeded_reset_already:
            seed = self.config.seed
            self._seeded_reset_already = True
        obs = self.env.reset(seed=seed)
        return to_tensor(obs, self.device).unsqueeze(0)

    def collect_rollouts(self) -> Tuple[TrajectoryBatch, float, float]:
        """采样一整段 on-policy 轨迹，并计算对应的 value、GAE 与 returns。"""

        T = self.config.rollout_steps

        states = torch.zeros((1, T + 1, self.obs_dim), dtype=torch.float32, device=self.device)
        rewards = torch.zeros((1, T), dtype=torch.float32, device=self.device)
        not_dones = torch.zeros((1, T), dtype=torch.float32, device=self.device)
        actions = torch.zeros((1, T, self.act_dim), dtype=torch.float32, device=self.device)
        action_log_probs = torch.zeros((1, T), dtype=torch.float32, device=self.device)

        completed_episode_info: List[Tuple[float, float]] = []
        last_state = self._reset_env()
        states[:, 0, :] = last_state

        for t in range(T):
            with torch.no_grad():
                action_pds = self.policy_model(last_state)
                next_action = self.policy_model.sample(action_pds)
                next_action_log_prob = self.policy_model.get_loglikelihood(action_pds, next_action)

            next_obs, reward, done, info = self.env.step(next_action.squeeze(0).cpu().numpy())
            if done:
                completed_episode_info.append(info["done"])
                next_obs = self.env.reset()

            next_state = to_tensor(next_obs, self.device).unsqueeze(0)
            rewards[:, t] = reward
            not_dones[:, t] = 0.0 if done else 1.0
            actions[:, t, :] = next_action
            action_log_probs[:, t] = next_action_log_prob
            states[:, t + 1, :] = next_state
            last_state = next_state

        if len(completed_episode_info) > 0:
            info_array = np.array(completed_episode_info, dtype=np.float64)
            avg_episode_length = float(info_array[:, 0].mean())
            avg_episode_reward = float(info_array[:, 1].mean())
        else:
            avg_episode_length = -1.0
            avg_episode_reward = -1.0

        states = states[:, :-1, :]
        with torch.no_grad():
            values = self.value_model(states).squeeze(-1)

        advantages, returns = compute_advantages_and_returns(
            rewards=rewards,
            values=values,
            not_dones=not_dones,
            gamma=self.config.gamma,
            gae_lambda=self.config.gae_lambda,
        )

        batch = TrajectoryBatch(
            states=states,
            rewards=rewards,
            not_dones=not_dones,
            actions=actions,
            action_log_probs=action_log_probs,
            returns=returns,
            advantages=advantages,
            values=values,
            unrolled=False,
        ).unroll()

        return batch, avg_episode_reward, avg_episode_length

    def value_step(self, batch: TrajectoryBatch) -> torch.Tensor:
        """对 value network 执行多轮 minibatch 优化。"""

        with torch.no_grad():
            old_values = self.value_model(batch.states).squeeze(-1).detach()

        final_loss = torch.tensor(0.0, device=self.device)
        for _ in range(self.config.val_epochs):
            for selected in minibatch_splits(batch.states.shape[0], self.config.num_minibatches):
                chosen = torch.as_tensor(selected, dtype=torch.long, device=self.device)
                predicted_values = self.value_model(batch.states[chosen]).squeeze(-1)
                final_loss = value_loss_gae(
                    values=predicted_values,
                    advantages=batch.advantages[chosen],
                    not_dones=batch.not_dones[chosen],
                    old_values=old_values[chosen],
                    clip_val_eps=self.config.clip_val_eps,
                    value_clipping=self.config.value_clipping,
                )
                self.value_optimizer.zero_grad()
                final_loss.backward()
                self.value_optimizer.step()
        return final_loss.detach()

    def ppo_step(self, batch: TrajectoryBatch) -> Tuple[torch.Tensor, Dict[str, float]]:
        """执行 PPO 的 clipped surrogate 更新，并返回损失与诊断信息。"""

        with torch.no_grad():
            old_pds = select_prob_dists(self.policy_model(batch.states), detach=True)

        final_loss = torch.tensor(0.0, device=self.device)
        for _ in range(self.config.ppo_epochs):
            for selected in minibatch_splits(batch.states.shape[0], self.config.num_minibatches):
                chosen = torch.as_tensor(selected, dtype=torch.long, device=self.device)

                dist = self.policy_model(batch.states[chosen])
                new_log_probs = self.policy_model.get_loglikelihood(dist, batch.actions[chosen])

                unclipped_reward = surrogate_reward(
                    batch.advantages[chosen],
                    new_log_probs=new_log_probs,
                    old_log_probs=batch.action_log_probs[chosen],
                )
                clipped_reward = surrogate_reward(
                    batch.advantages[chosen],
                    new_log_probs=new_log_probs,
                    old_log_probs=batch.action_log_probs[chosen],
                    clip_eps=self.config.clip_eps,
                )

                entropy_bonus = self.policy_model.entropies(dist).mean()
                surrogate = -torch.min(unclipped_reward, clipped_reward).mean()
                entropy_term = -self.config.entropy_coeff * entropy_bonus
                final_loss = surrogate + entropy_term

                self.policy_optimizer.zero_grad()
                final_loss.backward()
                if self.config.clip_grad_norm != -1.0:
                    torch.nn.utils.clip_grad_norm_(self.policy_model.parameters(), self.config.clip_grad_norm)
                self.policy_optimizer.step()

        diagnostics = compute_policy_diagnostics(
            policy=self.policy_model,
            states=batch.states,
            actions=batch.actions,
            old_log_probs=batch.action_log_probs,
            old_pds=old_pds,
        )
        return final_loss.detach(), diagnostics

    def trpo_step(self, batch: TrajectoryBatch) -> Tuple[torch.Tensor, Dict[str, float]]:
        """执行 TRPO 的自然梯度、KL 约束和回溯线搜索更新。"""

        initial_parameters = flatten_parameters(self.policy_model.parameters()).clone()
        pds = self.policy_model(batch.states)
        old_pds = select_prob_dists(pds, detach=True)
        action_log_probs = self.policy_model.get_loglikelihood(pds, batch.actions)

        surrogate_objective = surrogate_reward(
            batch.advantages,
            new_log_probs=action_log_probs,
            old_log_probs=batch.action_log_probs,
        ).mean()
        grad = torch.autograd.grad(surrogate_objective, self.policy_model.parameters(), retain_graph=True)
        flat_grad = flatten_parameters(grad)

        num_samples = int(batch.states.shape[0] * self.config.fisher_frac_samples)
        if num_samples <= 0:
            raise ValueError("fisher_frac_samples produced zero samples.")
        selected = np.random.choice(np.arange(batch.states.shape[0]), num_samples, replace=False)

        detached_selected_pds = select_prob_dists(pds, selected, detach=True)
        selected_pds = select_prob_dists(pds, selected, detach=False)
        kl = self.policy_model.calc_kl(detached_selected_pds, selected_pds).mean()
        g = flatten_parameters(
            torch.autograd.grad(kl, self.policy_model.parameters(), create_graph=True)
        )

        def fisher_product(x: torch.Tensor, damp_coef: float = 1.0) -> torch.Tensor:
            """构造 Fisher-vector product，用于共轭梯度近似求解 TRPO 步方向。"""

            z = g @ x
            hv = torch.autograd.grad(z, self.policy_model.parameters(), retain_graph=True)
            hv_flat = flatten_parameters(hv).detach()
            return hv_flat + x * self.config.damping * damp_coef

        step_direction = conjugate_gradient(fisher_product, flat_grad, self.config.cg_steps)
        denominator = step_direction @ fisher_product(step_direction)
        max_step_coeff = torch.sqrt(
            2.0 * torch.as_tensor(self.current_max_kl, device=self.device) / denominator
        )
        max_trpo_step = max_step_coeff * step_direction

        with torch.no_grad():
            def backtrack_fn(step_candidate: torch.Tensor) -> float:
                """评估某个候选 TRPO 步长是否同时满足改进和 KL 约束。"""

                assign_parameters(initial_parameters + step_candidate, self.policy_model.parameters())
                test_pds = self.policy_model(batch.states)
                test_log_probs = self.policy_model.get_loglikelihood(test_pds, batch.actions)
                new_reward = surrogate_reward(
                    batch.advantages,
                    new_log_probs=test_log_probs,
                    old_log_probs=batch.action_log_probs,
                ).mean()
                if (
                    new_reward <= surrogate_objective
                    or self.policy_model.calc_kl(pds, test_pds).mean() > self.current_max_kl
                ):
                    return -float("inf")
                return float((new_reward - surrogate_objective).item())

            expected_improve = flat_grad @ max_trpo_step
            final_step = backtracking_line_search(
                backtrack_fn,
                max_trpo_step,
                expected_improve,
                num_tries=self.config.max_backtrack,
            )
            assign_parameters(initial_parameters + final_step, self.policy_model.parameters())

        diagnostics = compute_policy_diagnostics(
            policy=self.policy_model,
            states=batch.states,
            actions=batch.actions,
            old_log_probs=batch.action_log_probs,
            old_pds=old_pds,
        )
        return surrogate_objective.detach(), diagnostics

    def train_step(self, iteration: int) -> Dict[str, float]:
        """完成一次完整训练迭代：采样、更新 value、更新 policy，并记录标量日志。"""

        rollout_batch, mean_reward, mean_length = self.collect_rollouts()
        value_loss = self.value_step(rollout_batch)

        self.current_max_kl += self.max_kl_increment

        if self.config.mode == "ppo":
            policy_metric, diagnostics = self.ppo_step(rollout_batch)
        elif self.config.mode == "trpo":
            policy_metric, diagnostics = self.trpo_step(rollout_batch)
        else:
            raise ValueError(f"Unsupported mode: {self.config.mode}")

        if self.config.anneal_lr:
            self.policy_scheduler.step()
            self.value_scheduler.step()

        log_row = {
            "iteration": float(iteration),
            "mean_reward": float(mean_reward),
            "mean_episode_length": float(mean_length),
            "policy_metric": float(policy_metric.item()),
            "value_loss": float(value_loss.item()),
            "mean_std": float(torch.exp(self.policy_model.log_stdev).mean().item()),
            "avg_kl": float(diagnostics["avg_kl"]),
            "max_ratio": float(diagnostics["max_ratio"]),
            "policy_lr": float(self.policy_optimizer.param_groups[0]["lr"]),
            "value_lr": float(self.value_optimizer.param_groups[0]["lr"]),
            "current_max_kl": float(self.current_max_kl),
        }
        self.training_log.append(log_row)
        return log_row

    def train(self, verbose_every: int = 10) -> List[Dict[str, float]]:
        """循环执行多次训练迭代，并按给定频率打印进度。"""

        start_time = time.time()
        for iteration in range(self.config.train_steps):
            row = self.train_step(iteration)
            if verbose_every > 0 and (iteration % verbose_every == 0 or iteration == self.config.train_steps - 1):
                print(
                    f"[{iteration:04d}/{self.config.train_steps}] "
                    f"reward={row['mean_reward']:.3f} "
                    f"len={row['mean_episode_length']:.1f} "
                    f"policy_metric={row['policy_metric']:.6f} "
                    f"value_loss={row['value_loss']:.6f} "
                    f"avg_kl={row['avg_kl']:.6f} "
                    f"max_ratio={row['max_ratio']:.6f}"
                )
        print(f"Training finished in {time.time() - start_time:.2f} seconds.")
        return self.training_log


## 10. 实验分析工具

训练完之后，你通常至少需要三类工具：

- 把日志字段转成数组
- 绘制单次实验曲线
- 对多 seed 结果做均值与置信区间估计

论文本身是基于多次运行统计结果讨论实现差异的，因此多 seed 汇总不是可有可无的附属功能，而是复现流程的一部分。


In [ ]:
def history_to_array(history: List[Dict[str, float]], key: str) -> np.ndarray:
    """从训练日志中抽取某个标量字段，转换成便于绘图的数组。"""

    return np.asarray([row[key] for row in history], dtype=np.float64)


def plot_history(history: List[Dict[str, float]], key: str = "mean_reward", title: Optional[str] = None) -> None:
    """绘制单次实验训练曲线，用于观察奖励或其它标量的变化趋势。"""

    values = history_to_array(history, key)
    plt.figure(figsize=(8, 4))
    plt.plot(values, linewidth=2)
    plt.xlabel("Training iteration")
    plt.ylabel(key)
    plt.title(title or key)
    plt.grid(alpha=0.3)
    plt.show()


def bootstrap_confidence_interval(
    samples: np.ndarray,
    num_bootstrap: int = 2000,
    alpha: float = 0.05,
    seed: int = 0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """通过 bootstrap 重采样估计多 seed 曲线的均值与置信区间。"""

    rng = np.random.default_rng(seed)
    bootstrap_means = []
    for _ in range(num_bootstrap):
        indices = rng.integers(0, samples.shape[0], size=samples.shape[0])
        bootstrap_means.append(samples[indices].mean(axis=0))
    bootstrap_means = np.asarray(bootstrap_means)
    mean = samples.mean(axis=0)
    lower = np.quantile(bootstrap_means, alpha / 2.0, axis=0)
    upper = np.quantile(bootstrap_means, 1.0 - alpha / 2.0, axis=0)
    return mean, lower, upper


def plot_seed_sweep(
    histories: List[List[Dict[str, float]]],
    key: str = "mean_reward",
    label: str = "experiment",
    alpha: float = 0.05,
) -> None:
    """绘制多 seed 实验的平均曲线及 bootstrap 置信区间。"""

    stacked = np.asarray([history_to_array(history, key) for history in histories], dtype=np.float64)
    mean, lower, upper = bootstrap_confidence_interval(stacked, alpha=alpha)
    x = np.arange(stacked.shape[1])
    plt.figure(figsize=(8, 4))
    plt.plot(x, mean, linewidth=2, label=label)
    plt.fill_between(x, lower, upper, alpha=0.25)
    plt.xlabel("Training iteration")
    plt.ylabel(key)
    plt.title(f"{label}: mean ± {(1 - alpha) * 100:.0f}% bootstrap CI")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


def run_seed_sweep(
    preset_name: str,
    seeds: Sequence[int],
    verbose_every: int = 50,
    **overrides,
) -> List[List[Dict[str, float]]]:
    """按多个随机种子重复运行实验，收集论文式的多次独立训练结果。"""

    histories = []
    for seed in seeds:
        config = make_config(preset_name, seed=seed, **overrides)
        print(f"Running {preset_name} with seed={seed}")
        trainer = ImplementationMattersTrainer(config)
        histories.append(trainer.train(verbose_every=verbose_every))
    return histories


## 11. 开始实验设置

到这里为止，构成训练系统的所有核心组件已经准备好了。  
下一步是正式设定“这次到底跑哪个实验”。

建议第一次学习时这样做：

- 先用 `walker_ppo`
- 把 `train_steps` 暂时改小，例如 `5` 或 `10`
- 确认整个训练循环逻辑正确，再恢复论文规模

这样你会更容易在 notebook 里观察每一步输出，而不是一上来就跑完整个实验。


In [ ]:
config = make_config("walker_ppo", seed=0, device="cpu")
print_config(config)


## 12. 开始单次训练

下面这个单元是**正式单次训练入口**。  
它会：

- 根据 `config` 创建训练器
- 在 `Walker2d-v4` 中采样 rollout
- 用 GAE 构造训练目标
- 更新 value 与 policy
- 记录并绘制 `mean_reward`

推荐学习方式：

- 第一次先把 `train_steps` 改成很小的值
- 跑通之后，再恢复你想比较的默认配置
- 观察 `avg_kl`、`max_ratio` 和 `value_loss` 是否处在合理范围


In [ ]:
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    trainer = ImplementationMattersTrainer(config)
    history = trainer.train(verbose_every=10)
    plot_history(history, key="mean_reward", title="Walker2d-v4 / PPO / mean_reward")
else:
    print("单次训练默认关闭。确认 Gymnasium / MuJoCo 运行时准备好后，将 RUN_FULL_TRAINING 设为 True。")


## 13. 开始多 Seed 训练

论文里的结果不是只靠一次幸运运行给出的，而是通过多次独立训练后做统计比较。  
因此，如果你要写课程项目报告，建议最终至少补一个多 seed 实验单元。

下面这个单元会：

- 对同一个预设反复运行多个随机种子
- 收集每个 seed 的训练曲线
- 用 bootstrap 画出均值与置信区间


In [ ]:
RUN_SEED_SWEEP = False

if RUN_SEED_SWEEP:
    sweep_histories = run_seed_sweep(
        "walker_ppo",
        seeds=[0, 1, 2],
        device="cpu",
        verbose_every=25,
    )
    plot_seed_sweep(sweep_histories, key="mean_reward", label="walker_ppo")
else:
    print("多 seed 训练默认关闭。需要复现实验统计结果时，将 RUN_SEED_SWEEP 设为 True。")


## 14. 自检清单

在真正把结果写进课程报告之前，建议做最后一遍自检：

1. 是否仍然是 `Walker2d-v4`，没有被切换成别的环境版本？
2. 是否只使用了 `gymnasium`，没有再回退到旧版 `gym`？
3. 是否真的没有导入原仓库的自定义模块？
4. `walker_ppo` 的关键超参数是否与 notebook 当前设定一致？
5. 是否明确区分了单次训练曲线与多 seed 汇总结果？
6. 如果你修改了超参数，是否在报告里明确说明这是你自己的实验设置？


In [ ]:
STEP_BY_STEP_AUDIT = {
    "imports_local_repo_modules": False,
    "has_actor": "CtsPolicy" in globals(),
    "has_critic": "ValueDenseNet" in globals(),
    "has_env_wrapper": "WalkerEnv" in globals(),
    "has_trainer": "ImplementationMattersTrainer" in globals(),
    "default_env_id": CONFIG_PRESETS["walker_ppo"].env_id,
    "default_env_is_walker2d_v4": CONFIG_PRESETS["walker_ppo"].env_id == "Walker2d-v4",
    "walker_ppo_lr": CONFIG_PRESETS["walker_ppo"].ppo_lr_adam,
    "walker_ppo_val_lr": CONFIG_PRESETS["walker_ppo"].val_lr,
    "walker_ppo_value_clipping": CONFIG_PRESETS["walker_ppo"].value_clipping,
}

assert STEP_BY_STEP_AUDIT["default_env_is_walker2d_v4"]
assert abs(STEP_BY_STEP_AUDIT["walker_ppo_lr"] - 4e-4) < 1e-12
assert abs(STEP_BY_STEP_AUDIT["walker_ppo_val_lr"] - 3e-4) < 1e-12
assert STEP_BY_STEP_AUDIT["walker_ppo_value_clipping"] is True

STEP_BY_STEP_AUDIT
